# Transformer Anomaly Detection

Detect and explain anomalies in transformer load data using local AI inference.

**Prerequisites:** Ollama running with `qwen2.5:3b` pulled. Install: `pip install requests pandas matplotlib`

**Note:** This notebook uses synthetic data. In production on Mini B, replace the data generation with your actual SCADA historian connection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from datetime import datetime, timedelta

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"

## Generate Synthetic SCADA Data

Simulate 7 days of transformer readings with injected anomalies.

In [ ]:
np.random.seed(42)
hours = 7 * 24
timestamps = [datetime(2026, 3, 1) + timedelta(hours=i) for i in range(hours)]
hour_of_day = np.array([t.hour for t in timestamps])

# Load follows daily cycle, temp correlates with load + ambient
daily_load = 45 + 30 * np.sin((hour_of_day - 6) * np.pi / 12)
load_pct = np.clip(daily_load + np.random.normal(0, 3, hours), 10, 100)
ambient_temp_c = 20 + 10 * np.sin((hour_of_day - 14) * np.pi / 12) + np.random.normal(0, 2, hours)
transformer_temp_c = ambient_temp_c + load_pct * 0.5 + np.random.normal(0, 1.5, hours)

# Anomaly 1: Cooling failure (hours 50-53)
transformer_temp_c[50:54] += 25
# Anomaly 2: Sustained overload (hours 100-108)
load_pct[100:109] = np.random.uniform(95, 99, 9)
transformer_temp_c[100:109] = ambient_temp_c[100:109] + load_pct[100:109] * 0.6 + 10
# Anomaly 3: Temperature-load decorrelation (hours 130-135)
transformer_temp_c[130:136] += 15

df = pd.DataFrame({
    'timestamp': timestamps, 'transformer_id': 'T-101',
    'load_pct': np.round(load_pct, 1),
    'transformer_temp_c': np.round(transformer_temp_c, 1),
    'ambient_temp_c': np.round(ambient_temp_c, 1),
})
print(f'Generated {len(df)} readings for T-101')
df.head(10)

## Visualize the Data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df['timestamp'], df['load_pct'], color='steelblue', linewidth=0.8)
axes[0].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
axes[0].set_ylabel('Load (%)')
axes[0].set_title('Transformer T-101 -- 7-Day Readings')
axes[0].legend()

axes[1].plot(df['timestamp'], df['transformer_temp_c'], color='orangered', linewidth=0.8, label='Transformer')
axes[1].plot(df['timestamp'], df['ambient_temp_c'], color='gray', linewidth=0.8, alpha=0.6, label='Ambient')
axes[1].set_ylabel('Temperature (C)')
axes[1].legend()

expected = df['ambient_temp_c'] + df['load_pct'] * 0.5
residual = df['transformer_temp_c'] - expected
axes[2].plot(df['timestamp'], residual, color='purple', linewidth=0.8)
axes[2].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Anomaly threshold')
axes[2].axhline(y=-10, color='red', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Temp Residual (C)')
axes[2].legend()

plt.tight_layout()
plt.show()

## Detect Anomalies

In [ ]:
expected_temp = df['ambient_temp_c'] + df['load_pct'] * 0.5
df['temp_residual'] = df['transformer_temp_c'] - expected_temp
threshold = df['temp_residual'].mean() + 2 * df['temp_residual'].std()

df['anomaly'] = (df['temp_residual'] > threshold) | (df['load_pct'] > 90)
anomalies = df[df['anomaly']].copy()
print(f'Detected {len(anomalies)} anomalous readings (threshold: {threshold:.1f}C residual)')
anomalies[['timestamp', 'load_pct', 'transformer_temp_c', 'ambient_temp_c', 'temp_residual']]

## Ask the LLM to Explain the Anomalies

In [ ]:
summary = anomalies[['timestamp', 'load_pct', 'transformer_temp_c', 'ambient_temp_c', 'temp_residual']].to_string(index=False)

prompt = f"""You are a power systems engineer analyzing transformer monitoring data.

Transformer T-101 (rated 50 MVA, oil-cooled) showed these anomalous readings over 7 days:

{summary}

Analyze these anomalies and provide:
1. A brief description of each anomaly cluster (group by time proximity)
2. Likely root cause for each cluster
3. Recommended actions ranked by urgency
4. Whether any warrant an immediate dispatch"""

response = requests.post(OLLAMA_URL, json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': prompt}],
    'stream': False,
})
print(response.json()['message']['content'])

## Summary

This notebook demonstrated:
1. Loading transformer SCADA data (synthetic here, real in production)
2. Statistical anomaly detection using temperature-load correlation residuals
3. Natural language explanation of anomalies via local LLM
4. Actionable maintenance recommendations

In the FERCoff sandbox, this entire workflow runs on Mini B with real CEII-protected data. The LLM never sees the internet. The data never leaves the air-gapped node.